## Δημήτριος Μουταφτσίδης 
##### moutdimi@ece.auth.gr

##### 31-05-25

In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import itertools
from pprint import pprint
import pandas as pd

# Πρόβλημα 1:

### Περιγραφή και παραδοχές: 
Πρόκειται για πρόβλημα εύρεσης feasible solution για εξαγωγή προγράμματος μαθημάτων σχολείου. Λύνουμε για feasibility , όχι optimality. Ο παράγοντας βελτιστοποίησης δεν είναι πολύ δύσκολο να ενταχθεί στην λύση. Ωστόσο πρόκειται για κάτι το σχετικό. Κάποιος δικαίως μπορεί να θεωρεί ότι χρειάζεται βελτιστοποίηση όσον αφορά το idle time των καθηγητών. Άλλος μπορεί να θεωρήσει ως βέλτιστο πρόγραμμα το οποίο θέλει τους μαθητές να γυρνάνε όσο το δυνατόν νωρίτερα. Έχουν και οι δύο δίκιο. Στα πλαίσια αυτής της εργασίας λοιπόν θα περιοριστούμε στην (μη μοναδική) λύση εξαγωγής βιώσιμου προγράμματος. Για την κατάτμηση της πενθήμερης εβδομάδας θα θεωρήσουμε πέντε ήμερες με τέσσερα διαθέσιμα δίωρα έκαστη. Σύνολο είκοσι μοναδικά timeslots λοιπόν. Όσον αφορά τα μαθήματα που έχουν δύο καθηγητές, απλοποιούμε αρκετά την ανάλυση με την εξής παραδοχή: τα Μαθηματικά των κυρίων Λαθοπράξη και Αντιπαραγώγου αντίμετωπιζονται σαν ξεχωριστά μαθήματα (math1 και math2 αντίστοιχα). Ομοίως για τις Γυμναστικές των Μπρατσάκη και Τρεχαντούλας (PE1 και PE2). Εφόσον το καθένα διδάσκεται αποκλειστικά σε μία τάξη, το split αυτό δεν θα μας προκαλέσει πρόβλημα. Να σημειωθεί πως εφόσον κάθε session διδασκαλίας αντιστοιχεί σε δίωρο, η έννοια του χρόνου δεν μας απασχολέι πραγματικά. Τα αντιμετωπίζουμε σαν κένα ή πλήρη slots σε κάποιον πίνακα.
### Μετάφραση σε κώδικα: 
Στο παρακάτω cell εισάγουμε σαν λίστες τις δυό Τάξεις , τα Μαθήματα και τους Καθηγητές. Κάθε φορά που χρειάζεται να έχουμε αντιστοιχεία τιμής με κάποια άλλη, ενώνουμε τις (σε προσεγμένη σειρά) Python lists σε Python dictionaries. Περισσότερες λεπτομέρεις φαίνονται στα σχόλια. Μια μικρή εξαίρεση γίνεται για το πρώτο δίωρο της Δευτέρας. Εφόσον είναι "κλεισμένο" για μελέτη, πρόκειται για περιορισμό. Πράγματι θα το δούμε στο αντίστοιχο snippet περιορισμών και μοντελοποίησης παρακάτω. Ένα απλούστερο (και πιο τεμπέλικο) σκεπτικό, είναι να αγνοήσουμε πλήρως την ύπαρξη του και ανάγουμε το πρόβλημα σε δεκαεννέα timeslots. Υλοποιούμε και τα δύο. Παρόλο που η εισαγωγή του περιορισμού είναι αχρείαστη, επέλεξα να την αφήσω και σαν showcase σκεπτικού,αλλά και σαν redundancy. Ο Solver είναι έτσι κι αλλίως πολύ γρήγορος. Ούτε επιβαρύνεται, ούτε βλάπτεται η ακεραιότητα της λύσης. 

In [3]:
#Define each variable set as lists of string objects.

#Classes: 
classes = ['C1' , 'C2']

#Subjects:
subjects = ['English' , 'Biology' , 'HistoryGeography' , 'Math1' , 'Math2' , 'Science' , 'Philosophy' , 'PE1' , 'PE2']

#Teachers:
#The list is order with the subjects list so each subject is in the same position with 
#the corresponding teacher.
teachers = ['Gesmanti' , 'Insulina' , 'Chartoula' , 'Antiparagogos' , 'Lathopraxis' , 'Kirkofidou' , 'Platiazon' , 'Bratsakis' , 'Trechantoula']

#Mapping each teacher to their subject as a dictionary.
#We will use it later to enforce teacher availability.
who_teaches_what = dict(zip(teachers , subjects))

#Now map timeslots through iteration of days and times lists.
#We need to examine all possible scheduling options.
days = ['Mon' , 'Tue' , 'Wed' , 'Thu' , 'Fri']
times = ['08-10' , '10-12' , '14-16' , '16-18']

#This double loop creates a list of day-time tuples.
timeslots = []
for d in days:
    for t in times:
        #Exclude the first two hours of Monday.
        #Since no classes must be taught then we treat it as if it didn't exist.
        if not (d == 'Mon' and t == '08-10'):  #CONTROL
            timeslots.append((d , t))

#Now we create a list of class-to-subject pairings through Cartesian Product (set of all ordered pairs).
#We will combine each class-subject pair to the required hours.
pairs = list(itertools.product(classes , subjects))

#We declare the required number of teaching sessions per class , per subject.
#Again we pay attention to the order so that each index corresponds to the appropriate pair.
req_c1 = [1 , 3 , 2 , 4 , 0 , 3 , 1 , 1 , 0]
req_c2 = [1 , 3 , 2 , 0 , 4 , 3 , 1 , 0 , 1]
#Now merge (not add) them.
req_tot = req_c1 + req_c2

#Finally, zipping into a single dictionary for a unified requirement-to-(class , subject) pairing.
requirements = dict(zip(pairs , req_tot))

#### Σύντομοι έλεγχοι:
Μία γρήγορη εκτύπωση για να επιβεβαιώσουμε ότι έχουμε εισάγει τα δεδομένα όπως πρέπει.

In [5]:
#Quick checks to see if data mapping is correct.
#Using pprint saves us the trouble of formatted printing.
print("Teacher-Subject Correspondance.")
pprint(who_teaches_what)
print('\n')
print("Timetable.")
pprint(timeslots)
print('\n')
print("Required hours per class per subject.")
pprint(requirements)

#We visually validate the results to make sure they are en par with the assignment.

Teacher-Subject Correspondance.
{'Antiparagogos': 'Math1',
 'Bratsakis': 'PE1',
 'Chartoula': 'HistoryGeography',
 'Gesmanti': 'English',
 'Insulina': 'Biology',
 'Kirkofidou': 'Science',
 'Lathopraxis': 'Math2',
 'Platiazon': 'Philosophy',
 'Trechantoula': 'PE2'}


Timetable.
[('Mon', '10-12'),
 ('Mon', '14-16'),
 ('Mon', '16-18'),
 ('Tue', '08-10'),
 ('Tue', '10-12'),
 ('Tue', '14-16'),
 ('Tue', '16-18'),
 ('Wed', '08-10'),
 ('Wed', '10-12'),
 ('Wed', '14-16'),
 ('Wed', '16-18'),
 ('Thu', '08-10'),
 ('Thu', '10-12'),
 ('Thu', '14-16'),
 ('Thu', '16-18'),
 ('Fri', '08-10'),
 ('Fri', '10-12'),
 ('Fri', '14-16'),
 ('Fri', '16-18')]


Required hours per class per subject.
{('C1', 'Biology'): 3,
 ('C1', 'English'): 1,
 ('C1', 'HistoryGeography'): 2,
 ('C1', 'Math1'): 4,
 ('C1', 'Math2'): 0,
 ('C1', 'PE1'): 1,
 ('C1', 'PE2'): 0,
 ('C1', 'Philosophy'): 1,
 ('C1', 'Science'): 3,
 ('C2', 'Biology'): 3,
 ('C2', 'English'): 1,
 ('C2', 'HistoryGeography'): 2,
 ('C2', 'Math1'): 0,
 ('C2', 'Math2'

### Μοντελοποίηση , περιορισμοί και επίλυση:
Χρησιμοποιούμε ψευδοαντικειμενική συνάρτηση για την επίλυση του προβλήματος. Ζητάμε από τον Solver του Gurobipy να "ελαχιστοποιήσει το 0" υπό περιορισμούς ώστε να το "ξεγελάσουμε" στο να μας δώσει εφικτή λύση. Να επισημάνουμε εδώ ότι ο Solver απαντά με την πρώτη εφικτή λύση που συναντά. Παρακάτω θα δούμε πως δύναται να υπάρχει παραπάνω από μία. Για την μοντελοποίηση ορίζουμε Binary μεταβλητές απόφασης 
$x_{c,j,d,t} \; \in \; \{0,1\} $. 
Για να λάβει την τιμή 1, πρέπει η τάξη c να έχει μάθημα j την μέρα d στις t η ώρα. Σημειώνουμε πως $c \in \{C1,C2\}$, $j \in \mathcal{J}$, and $(d,t) \in \mathcal{S}$ , όπου $\mathcal{J}$ το σύνολο των μαθημάτων που ορίσαμε προηγουμένως και $\mathcal{S}$ το σύνολο των timeslots. $\{C1,C2\}$ είναι οι δύο τάξεις. Ας δούμε τώρα και τους περιορισμούς. Θα τους δούμε με την σειρά που εμφανίζονται στον κώδικα παρακάτω.
##### Περιορισμός 1:
Κάθε μάθημα πρέπει να διδάσκεται συγκερκιμένες ώρες την εβδομάδα. Ή μάλλον για κάθε timeslot , η κάθε τάξη έχει το κάθε μάθημα $r_{c,j}$ φορές (ο απαιτόυμενος αριθμός ωρών, όπως ορίζεται από την εκφώνηση). Ειδικότερα:
$$
\sum_{(d,t)\in \mathcal{S}}
\;x_{c,j,d,t}
\;=\; r_{c,j},
\quad 
\forall\, c\in\{C1,C2\},\; j\in \mathcal{J}.
$$
##### Περιορισμός 2:
Το πρώτο δίωρο της Δευτέρας δεν γίνονται μαθήματα. Ή μάλλον $x_{c,j,Mon,8-10} = 0$. Όπως είδαμε νωρίτερα αυτό το έχουμε χειριστεί ήδη, καθώς αυτό το timeslot δεν υπάρχει.
##### Περιορισμός 3:
Δεν γίνεται κάποια τάξη να κάνει ταυτόχρονα δύο μαθήματα. Ή μάλλον για κάθε timeslot και κάθε τάξη, το πολύ μία μεταβλητή παίρνει την τιμή 1 για όλα τα μαθήματα. Ειδικότερα: 
$$
\sum_{j \in \mathcal{J}}
\;x_{c,j,d,t}
\;\le\; 1,
\quad
\forall\, c\in\{C1,C2\},\; (d,t)\in \mathcal{S}.
$$
##### Περιορισμός 4: 
Κάθε μάθημα δεν πρέπει να διδάσκεται πάνω από ένα δίωρο την μέρα. Ή μάλλον για κάθε τάξη , μάθημα και μέρα η μεταβλητή απόφασης παίρνει την τιμή 1 το πολύ μία φορά για όλα τα timeslots. ΕΙδικότερα:
$$
\sum_{t : (d,t)\in \mathcal{S}}
\;x_{c,j,d,t}
\;\le\; 1,
\quad
\forall\, c\in\{C1,C2\},\; j\in \mathcal{J},\; d\in \mathcal{D}.
$$
##### Περιορισμός 5: 
Οι καθηγητές δεν μπορούν να βρίσκονται σε δύο τάξεις ταυτόχρονα. Εδώ αρκεί να φροντίσουμε να μην διδάσκεται το ίδιο μάθημα στις δύο τάξεις ταυτόχρονα, αφού με τις παραδοχές μας υπάρχει μοναδική αντιστοιχία μαθήματος-καθηγητή. Είναι προσβάσιμη μέσω του dictionary who_teaches_what. Ειδικότερα: 
$$
\sum_{c \in \{C1,C2\}}
\;x_{\,c,\;\mathrm{who\_teaches\_what}[k],\,d,\,t}
\;\le\; 1,
\quad
\forall\, k \in \mathcal{K},\; (d,t)\in \mathcal{S}.
$$
##### Περιορισμός 6:
Η κ. Ινσουλίνη απουσιάζει τις Τετάρτες. Ή μάλλον η τιμή του x είναι μηδενική για κάθε τάξη και κάθε ώρα για ημέρα Τετάρτη και μάθημα Βιολογίας. Η 1-1 αντιστοίχιση πάλι μας βοηθάει. Ειδικότερα: 
$$
x_{\,c,\;\mathrm{Biology},\,\mathrm{Wed},\,t}
\;=\; 0,
\quad
\forall\, c\in\{C1,C2\},\; t\in \mathcal{T}.
$$
##### Περιορισμός 7: 
Ο κ. Λαθοπράξης απουσιάζει το πρώτο διδακτικό δίωρο της Δευτέρας. Είναι ίδιο σενάριο με το προηγούμενο, μόνο που αφορά ένα timeslot. Ειδικότερα: 
$$
x_{\,c,\;\mathrm{Math2},\,\mathrm{Mon},\,10\!-\!12}
\;=\; 0,
\quad
\forall\, c\in\{C1,C2\}.
$$
##### Περιορισμός 8:
Η Γυμναστική διδάσκεται συγκεκριμένες ώρες της εβδομάδας. Αυτό εμπεριέχει δύο υπόπεριορισμούς. Να διδάσκεται σίγουρα Πέμπτη 14:00 - 16:00 και να μην διδάσκεται ποτέ άλλοτε. Βέβαια αν αναλογιστούμε ότι οι απαιτούμενες ώρες αντιστοιχούν σε ένα δίωρο ανά τάξη, ο πρώτος θα αρκούσε. Αλλά λίγο redundancy δεν βλάπτει. Θυμόμαστε ότι το μάθημα Γυμναστική αντιστοιχεί σε ΡΕ1 και ΡΕ2. Ειδικότερα: 
$$
x_{\,C1,\,\mathrm{PE1},\,\mathrm{Thu},\,14\!-\!16}
\;=\; 1, 
\quad 
x_{\,C2,\,\mathrm{PE2},\,\mathrm{Thu},\,14\!-\!16} 
\;=\; 1,
\\
\quad
x_{\,c,\,\mathrm{PE}_\ell,\,d,\,t}
\;=\; 0,
\quad
\forall\,(c,\mathrm{PE}_\ell)\in\{(C1,\mathrm{PE1}),(C2,\mathrm{PE2})\}, 
\; (d,t)\in \mathcal{S}\setminus\{(\mathrm{Thu},\,14\!-\!16)\}.
$$
Το μοντέλο μας είναι πλέον πλήρες. Το Gurobi μας ενημερώνει πως βρήκε λύση. Τέτοιου είδους MILP προβλήματα λύνονται πολύ γρήγορα από τους Solvers. Στην περίπτωση μας μάλιστα, ενημερωνόμαστε για "zero simplex iterations". Εφόσον δηλαδή εισάγαμε σταθερή αντικειμενική συνάρτηση, το Gurobi βρίσκει μια εφικτή λύση η οποία είναι αυτομάτως και βέλτιστη. Οπότε δεν προχωράει σε βελτίωση των Bounds. 
### Μετάφραση σε κώδικα: 
Η επιβολή των περιορισμών είναι αρκετά απλή και επαναλαμβανόμενη διαδικασία. Έχει την μορφή πολλαπλών loops. Στο εσωτερικό τους καλείται η addConstr του Gurobi που επιβάλει τον κατάλληλο ισοτικό ή ανισοτικό περιοριοσμό. Για τον υπολογισμό αθροίσματος καλείται η quicksum. Είναι γρηγορότερη από την sum() της standard Python και Optimized για OR. Περισσότερες λεπτομέρειες στα σχόλια. Να σημειωθεί ότι το trick του διαχωρισμού των μαθημάτων και η (αισίως) 1-1 αντιστοιχίση μας επιτρέπει να αποφύγουμε Looping της λίστας καθηγητών. Την μία φορά που απαιτείται (στον περιορισμό 5), χρησιμοποιούμε το who_teaches_what dictionary για να επιστρέψουμε στο σύνολο J. Η λογική, έτσι είναι αρκετά πιο απλή και intuitive.

In [10]:
#Now we proceed with Model creation.

#First and foremost, create the Gurobipy model object.
#It holds all the required information like variables, constraints and objectibves.
model = gp.Model('WeeklyTimetable')

#Declaring the varibles as binaries named x.
#The 0-1 values depend on wether class X has subject Y and day Z at V o'clock
x = model.addVars(classes , subjects , days , times , vtype=GRB.BINARY, name = 'x')

#Constraint No. 1.
#Enforce the needed hours for each subject.
#We loop through each class-subject tuple and match it with the exact number of sessions.
for (c , j), req in requirements.items():
    model.addConstr(gp.quicksum(x[c , j , d , t] for d , t in timeslots) == req, name = f"load_{c}_{j}")

#Constraint No. 2.
#Exclude the first two hours of Monday.
#Redundant, as we made sure earlier that it is not available for "booking".
#In case of slip though, we make sure it's zero.
for c in classes:
    for j in subjects:
        model.addConstr(x[c , j , 'Mon' , '08-10'] == 0 , name=f"noMonPrep_{c}_{j}")

#Constraint No. 3.
#Forbid simultaneous courses for each class.
#We loop through each class and timeslot and make it so that the sum is either 0 or 1.
for c in classes:
    for d , t in timeslots:
        model.addConstr(gp.quicksum(x[c , j , d ,t] for j in subjects) <= 1 , name = f"oneSubjPerSlot_{c}_{d}_{t}")

#Constraint No. 4.
#Make sure each subject is taught at most once per day.
#We quadra-loop through each class per each subject per each day per each time interval and enforce that the subject count <= 1.
for c in classes:
    for j in subjects:
        for d in days:
            model.addConstr(gp.quicksum(x[c , j , d , t] for t in times if (d , t) in timeslots) <= 1 , name=f"onePerDay_{c}_{j}_{d}")

#Constraint No. 5.
#No teacher shall be in two classes at the same time.
#Here we triple-loop each techer, timeslot and class and once again enforce the <= 1 constraint.
for k in teachers:
    subj = who_teaches_what[k]
    for d , t in timeslots:
        model.addConstr(gp.quicksum(x[c , subj , d , t] for c in classes) <= 1 , name = f"teachLoad_{k}_{d}_{t}")

#Constraint No. 6. 
#Mrs. Insulina cannot teach on a Wednesday.
#We zero-set all Biology courses each Wednesday.
for c in classes:
    for t in times:
        if ('Wed' , t) in timeslots:
            model.addConstr(x[c , 'Biology' , 'Wed' , t] == 0 , name=f"noBioWed_{c}_{t}")
                            
#Constraint No. 7.
#Lathopraxis cannot teach Monday 10:00-12:00.
#We set this timeslot to 0 for Lathopraxis (aka Math2 , since this is the only subject he teaches.)
for c in classes:
    if ('Mon','10-12') in timeslots:
        model.addConstr(x[c , 'Math2' , 'Mon' , '10-12'] == 0, name = f"noMath2Mon_{c}")

#Constraint No. 8.
#PE must be taught every Thursday 14:00 - 16:00.

#Thus, we set exactly that.
model.addConstr(x['C1' , 'PE1' , 'Thu' , '14-16'] == requirements[('C1' , 'PE1')] , name = "PE1_slot")
                
model.addConstr(x['C2' , 'PE2' , 'Thu' , '14-16'] == requirements[('C2' , 'PE2')] , name = "PE2_slot")

#And then exclude it for each and every other timeslot.
#Again, iterating through each class-PE pair and timeslot and setting it to 0.
for c , j in [('C1' , 'PE1') , ('C2' , 'PE2')]:
    for d , t in timeslots:
        if not (d == 'Thu' and t == '14-16'):
            model.addConstr(x[c , j , d , t] == 0 , name = f"noOtherPE_{c}_{j}_{d}_{t}")

#In order for the solver to process the problem we set a dummy objective (minimizing 0).
#We thus solve for feasibility, not optimality.
model.setObjective(0, GRB.MINIMIZE)
model.optimize()

#And now our work has ended. Time to present it.
#Pandas library has got our back.

Restricted license - for non-production use only - expires 2026-11-23
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 8840U w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 383 rows, 360 columns and 1434 nonzeros
Model fingerprint: 0x90f3ac71
Variable types: 0 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%


### Αποτελέσματα:
Το παρακάτω snippet μετατρέπει την λύση σε ευανάγνωστο dataframe. Γίνεται και αναγωγή σε ωρολόγιο πρόγραμμα του κάθε καθηγητή. Με οπτική επιθεώρηση επιβεβαιώνουμε πως οι περιορισμοί έχουν ικανοποιηθεί.

In [13]:
#If the pseudo-optimal (aka feasible solution) has been found:
#We loop through timeslots , clases , subjects and store the values
#in a human-reader-friendly Pandas dataframe.
if model.Status == GRB.OPTIMAL:
    rows = []
    for d , t in timeslots:
        for c in classes:
            scheduled = [j for j in subjects if x[c , j , d , t].X > 0.5]          
            rows.append({'Class': c , 'Day': d , 'Time': t , 'Subject': scheduled[0] if scheduled else ''})
                
    df = pd.DataFrame(rows)
    timetable = df.pivot_table(index=['Day','Time'],columns='Class',values='Subject',aggfunc='first').sort_index()

#If no optimality (aka feasibility) is achieved, we let the user know.
else:
    print("No feasible solution found.")

#Reindexing for proper presentation.
timetable = timetable.reindex(timeslots)

print("Class Timetable")
display(timetable)
print()

#Now manipulate the per-class schedule to turn it to teacher working hours.
#Again, looping through timeslots, teachers , classes and note at which class the
#corresponding teacher teaches at that particular time. Else, we leave it empty.
rows = []
for d , t in timeslots:
    row = {'Day': d , 'Time': t}
    for k in teachers:
        subj = who_teaches_what[k]
        taught = [c for c in classes if x[c , subj , d , t].X > 0.5]
        row[k] = taught[0] if taught else ''
    rows.append(row)

#Setting as Pandas dataframe and reindexing for readability.
df_teacher = pd.DataFrame(rows)
teacher_timetable = df_teacher.set_index(['Day','Time'])[teachers]
teacher_timetable = teacher_timetable.reindex(timeslots)

print("Teacher's Timetable")
display(teacher_timetable)

Class Timetable


Class                    C1                C2
Day Time                                     
Mon 10-12           English                  
    14-16           Biology             Math2
    16-18  HistoryGeography           Biology
Tue 08-10                               Math2
    10-12           Science                  
    14-16  HistoryGeography           Biology
    16-18             Math1  HistoryGeography
Wed 08-10                    HistoryGeography
    10-12                             Science
    14-16        Philosophy                  
    16-18             Math1        Philosophy
Thu 08-10           Biology             Math2
    10-12             Math1           Science
    14-16               PE1               PE2
    16-18           Science           English
Fri 08-10           Biology                  
    10-12           Science           Biology
    14-16                             Science
    16-18             Math1             Math2


Teacher's Timetable


Gesmanti Insulina Chartoula Antiparagogos Lathopraxis Kirkofidou  \
Day Time                                                                     
Mon 10-12       C1                                                           
    14-16                C1                                  C2              
    16-18                C2        C1                                        
Tue 08-10                                                    C2              
    10-12                                                               C1   
    14-16                C2        C1                                        
    16-18                          C2            C1                          
Wed 08-10                          C2                                        
    10-12                                                               C2   
    14-16                                                                    
    16-18                                        C1                          
Thu 08-10                C1                                  C2              
    10-12                                        C1                     C2   
    14-16                                                                    
    16-18       C2                                                      C1   
Fri 08-10                C1                                                  
    10-12                C2                                             C1   
    14-16                                                               C2   
    16-18                                        C1          C2              

          Platiazon Bratsakis Trechantoula  
Day Time                                    
Mon 10-12                                   
    14-16                                   
    16-18                                   
Tue 08-10                                   
    10-12                                   
    14-16                                   
    16-18                                   
Wed 08-10                                   
    10-12                                   
    14-16        C1                         
    16-18        C2                         
Thu 08-10                                   
    10-12                                   
    14-16                  C1           C2  
    16-18                                   
Fri 08-10                                   
    10-12                                   
    14-16                                   
    16-18

### Πρόσθετα: 
Μιλήσαμε προηγουμένως για μη μοναδικότητα λύσεων. Ο παρακάτω κώδικας τρέχει το μοντέλο πολλές φορές και κάθε αποκλείει την προηγούμενη λύση. Την θεωρεί ως constraint. Έτσι, θεωρητικά κάθε λύση που βλέπουμε είναι μοναδική. Μάλιστα η διαδικασία ελέγχεται εκ νέου. Στα αποτελέσματα φαίνεται ότι για 100 δοκιμές παίρνουμε 100 διαφορετικές λύσεις. Ο μέγιστος αριθμός δοκιμών που δοκιμάστηκε ήταν 1000, καθώς το license του Gurobi μπλοκάρει μεγαλύτερης κλίμακας δοκιμές. Πάλι είδαμε 1000 μοναδικές λύσεις. Ομολογώ πως ο αριθμός φαίνεται ύποπτα μεγάλος. Θεωρώ ωστόσο πως ο κώδικας ελέγχου είναι ορθός. Οπότε πράγματι έχουμε τουλάχιστον 1000 διαφορετικές λύσεις. Ενδεικτικά, επιθεωρούμε οπτικά και δύο από τις λύσεις. Πράγματι διαφέρουν.


#### Σημείωση: 
Αφήνουμε ανοιχτό το ενδεχόμενο ότι ως Δευτέρα πρωί για τον κ. Λαθοπράξη θεωρούμε την ήδη νεκρή ζώνη 08:00 - 10:00. Στην παρούσα εργασία θεωρούμε ότι αφορά και την ζώνη 10:00 - 12:00. Αν δεν είναι έτσι, τότε το μόνο που έχουμε να κάνουμε είναι να αναιρέσουμε τον αντιστοίχο περιορισμό, αφού επικαλύπτεται πλήρως από αυτόν της πρωινής μελέτης. 

Για βελτιστοποίηση προγράμματος (πχ. αποφυγή πρωινών μαθημάτων για τους μαθητές) μπορούμε να εισάγουμε συνάρτηση κόστους με βάρη στις πρωινές ζώνες. Έτσι ο αλγόριθμος βελτιστοποίησης αποθαρρύνεται από το να τις επιλέξει.

In [39]:
#Making the solver non-verbose.
model.Params.OutputFlag = 0

#This will be a list of timetables.
distinct_solutions = []
max_solutions = 100

#This loop runs max_solutions times. 
#Each time, optimize is called using our initial pseudo-objective function. 
#The variable assignments are stored in a set. The distinct_solutions is checked.
#If the current solution turns out to be a reapeated one, we terminate the loop.
#If not, we append the solution to the distinct_solutions lists and continue.
while len(distinct_solutions) < max_solutions:
    model.optimize()
    #If no more solutions can be found the loop brakes.
    if model.Status != GRB.OPTIMAL:
        break
    current_set = set()
    for d, t in timeslots:
        for c in classes:
            for j in subjects:
                if x[c, j, d, t].X > 0.5:
                    current_set.add((c, d, t, j))

    if any(current_set == s for s in distinct_solutions):
        break

    distinct_solutions.append(current_set)

    #Most recent solution is added as a constraint so we don't encounter it again.
    ng = gp.quicksum(x[c, j, d, t] for (c, d, t, j) in current_set) <= len(current_set) - 1
    model.addConstr(ng)

print(f"{len(distinct_solutions)} distinct solutions found")

#Additional checking through frozen set.
patterns = [frozenset(sol) for sol in distinct_solutions]

num_unique = len(set(patterns))
num_total = len(patterns)

if num_unique == num_total:
    print(f"All {num_total} solutions are genuinely distinct.")
else:
    print(f"{num_total} solutions found, but only {num_unique} unique patterns.")

#Finally we convert to dataframe and print two of the solutions to visually verify.
def timetable_from_set(assignment_set):
    rows = []
    for d, t in timeslots:
        for c in classes:
            subj = ''
            for (cc, dd, tt, jj) in assignment_set:
                if cc == c and dd == d and tt == t:
                    subj = jj
                    break
            rows.append({'Class': c, 'Day': d, 'Time': t, 'Subject': subj})
    df = pd.DataFrame(rows)
    tt = df.pivot_table(index=['Day','Time'], columns='Class', values='Subject', aggfunc='first').sort_index()
    return tt.reindex(timeslots)

if len(distinct_solutions) >= 2:
    print("--- Solution #1 ---")
    print(timetable_from_set(distinct_solutions[0]))
    print("\n--- Solution #2 ---")
    print(timetable_from_set(distinct_solutions[1]))
else:
    print("Fewer than 2 solutions found.")

100 distinct solutions found
All 100 solutions are genuinely distinct.
--- Solution #1 ---
Class                    C1                C2
Day Time                                     
Mon 10-12           Biology           Science
    14-16             Math1                  
    16-18  HistoryGeography           Biology
Tue 08-10                                    
    10-12           English           Science
    14-16             Math1           Biology
    16-18           Science             Math2
Wed 08-10                             Science
    10-12             Math1  HistoryGeography
    14-16  HistoryGeography                  
    16-18           Science             Math2
Thu 08-10        Philosophy             Math2
    10-12           Science        Philosophy
    14-16               PE1               PE2
    16-18           Biology  HistoryGeography
Fri 08-10           Biology           English
    10-12                             Biology
    14-16             Math1        

# Πρόβλημα 2:

### Περιγραφή και παραδοχές:
Πρόκειται για πρόβλημα ελαχιστοποίησης κόστους διανομής. Αποθήκες πεπερασμένης χωρητικότητας με fixed, one time έξοδα κατασκευής προμηθεύουν κέντρα διανομής. Η μεταφορά έχει και αυτή κόστος ανάλογο του συνολικού φορτίου. Τα κέντρα διανομής πρέπει να ικανοποιούνται πλήρως. Οι αποθήκες δεν είναι απαραίτητο να λειτουργήσουν. Καλούμαστε λοιπόν να αποφασίσουμε το ποιές αποθήκες θα ανοίξουν. Με τις παρούσες παραδοχές, επιπλέον, καλούμαστε να υποδείξουμε το πόσο φορτίο θα μεταφέρεται απο τις αποθήκες και το σε ποιά κέντρα. Στόχος είναι να βρεθεί ο βέλτιστος συνδυασμός τοποθεσιών για αποθήκες και το βέλτιστο cargo flow ώστε να έχουμε τα λιγότερα δυνατά έξοδα. Σημειώνουμε πως μερικές διαδρομές είναι αδύνατες, γεγονός που πρέπει να το σεβαστούμε. Δεν θα προκύψει ,ώστόσο, ως περιορισμός στο μοντέλο. Όπως και προηγουμένως, θα προσποιηθούμε ότι οι διαδρομές αυτές δεν υπάρχουν.

### Μετάφραση σε κώδικα: 
Παρακάτω εισάγουμε τα απαραίτητα δεδομένα με την μορφή dictionary. Η απευθείας χρήση τους και η αποφυγή zips (όπως πριν) μας γλίτωσε από πιθανά λάθη. Οι τιμές είναι πολλές και χρειάστηκε αρκετή προσοχή για να περαστούν. Τα per-tonne κόστη θα αποτελέσουν κριτήριο feasibility της εκάστοτε διαδρομής. Γι'αυτό και φροντίζουμε να περιλαμβάνουν μόνο εφικτά ζεύγη αποθηκών-κέντρων. Οι υπόλοιπες λεπτομέρειες φαίνονται στα σχόλια.

In [54]:
#We first declare two lists carrying warehouse and distribution center IDs.
warehouses = list(range(1 , 13))
centers = list(range(1 , 13))

#We now correspond each warehouse to its fixed cost and capacity through two dictionaries.
F = {1 : 3500 , 2 : 9000 , 3 : 10000 , 4 : 4000, 5 : 3000 , 6 : 9000 , 7 : 9000 , 8 : 3000 , 9 : 4000 , 10 : 10000 , 11 : 9000 , 12 : 3500}
     
K = {1 : 300 , 2 : 250, 3 : 100 , 4 : 180 , 5 : 275 , 6 : 300 , 7 : 200 , 8 : 220 , 9 : 270 , 10 : 250 , 11: 230 , 12 : 180}

#The distribution centers are also matched to their demands through a dictionary.
D = {1 : 120 , 2 : 80 , 3 : 75 , 4 : 100 , 5 : 110 , 6 : 100 , 7 : 90 ,  8 : 60 , 9 : 30 , 10 : 150 , 11 : 95 , 12 : 120}
     

#T contains the total transport costs to serve the demand of center j from warehouse i.
#It is a complex stratcure. First we painstakingly enter the total cost as given in the assignment.
#Then we correspond each warehouse-center tuple to a value by iterating through dictionary keys (warehouse)
#and values (demand per center). ENumeration yields the distribution center index.
T = {
  (i , j) : val
  for i , row in {
    1 : [100 , 80 , 50 , 50 , 60 , 100 , 120 , 90 , 60 , 70 , 65 , 110],
    2 : [120 , 90 , 60 , 70 , 65 , 110 , 140 , 110 , 80 , 80 , 75 , 130],
    3 : [140 , 110 , 80 , 80 , 75 , 130 , 160 , 125 , 100 , 100 , 80 , 150],
    4 : [160 , 125 , 100 , 100 , 80 , 150 , 190 , 150 , 130 , float('inf') , float('inf') , float('inf')],
    5 : [190 , 150 , 130 , float('inf') , float('inf') , float('inf') , 180 ,150 ,50 ,50 ,60 , 100],
    6 : [200 ,180 ,150 ,float('inf'), float('inf') , float('inf') , 100 , 120 ,90 , 60 , 75 , 110],
    7 : [120 , 90 , 60 , 70 , 65 , 110 , 140 , 110 , 80 , 80 , 75 , 130],
    8 : [120 , 90 , 60 , 70 , 65 , 110 , 140 , 110 , 80 , 80 , 75 , 130],
    9 : [140 , 110 , 80 , 80 , 75 , 130 , 160 , 125 , 100 , 100 , 80 , 150],
    10 : [160 , 125 , 100 , 100 , 80 , 150 , 190 , 150 , 130 , float('inf') , float('inf') , float('inf')],
    11 : [190 , 150 , 130 , float('inf') , float('inf') , float('inf') , 200 , 180 , 150, float('inf') , float('inf') , float('inf')],
    12 : [200 , 180 , 150 , float('inf') , float('inf') , float('inf') , 100 , 80 , 50 , 50 , 60 , 100],
  }.items()
  for j , val in enumerate(row , start = 1)
}

#We now calculate the per tonne costs. 
#Infisible routes are excluded here. They shall not be a problem from here onward.
c = {(i , j) : T[i , j] / D[j] for i , j in T if T[i , j] < float('inf')}

#Now we are ready to proceed to modeling.

# Μοντελοποίηση , περιορισμοί και επίλυση: 
Ως αντικειμενική συνάρτηση ορίζουμε το συνολικό άθροισμα κόστους κατασκευής των λειτουργικών αποθηκών, μαζί με το συνολικό κόστος μεταφοράς προϊόντος από την εκάστοτε αποθήκη στα κέντρα που εξυπηρετεί. Tο κόστος μεταφοράς ,συγκεkριμένα, το ορίζουμε ως την μεταφερόμενη ποσότητα σε τόνους επί το κόστος/τόνο της εκάστοτε διαδρομής. Συγκεκριμένα:
$$
\min \quad 
\sum_{i=1}^{12} F_i\,y_i 
\;+\; 
\sum_{(i,j)\in \mathcal{C}} c_{i,j}\,f_{i,j}
$$
όπου: 
- $y_i \in \{0,1\}$ δείχνει αν θα λειτουργήσει η αποθήκη i.  
- $F_i$ είναι το πάγιο κόστος κατασκευής (χιλ. €) για την αποθήκη i.  
- $f_{i,j}\ge 0$ είναι οι τόνοι που στέλνονται από την αποθήκη i στο κέντρο j.  
- $c_{i,j} = \frac{T_{i,j}}{D_j}$ είναι το κόστος (χιλ. €) ανά τόνο από την i προς το j.  
- $\mathcal{C} = \{(i,j)\mid T_{i,j}<\infty\}$ είναι το σύνολο των επιτρεπτών ζευγών i,j.

Ελαχιστοποιούμε υπό τους ακόλουθους περιορισμούς.

##### Περιορισμός 1: 
Η αποθήκη δεν μπορεί να προμηθεύει παραπάνω από το capacity. Ή μάλλον για κάθε αποθήκη, το άθροισμα της ποσοτητας μεταφερόμενου φορτίου για κάθε δυνατή διαδρομή δεν μπορεί να υπρβαίνει την χωρητικότητά της.Συγκεκριμένα: 
$$
\sum_{\,j : (i,j)\in \mathcal{C}} f_{i,j} \;\le\; K_i\,y_i,
\quad \forall\,i \in \{1,\ldots,12\}
$$
όπου:
- $f_{i,j}$ είναι οι τόνοι που στέλνονται από την αποθήκη i στο κέντρο j.  
- $K_i$ είναι η χωρητικότητα (σε τόνους) της αποθήκης i.  
- $y_i \in \{0,1\}$ δείχνει εάν η αποθήκη i ανοίγει ($y_i=1$) ή όχι ($y_i=0$).  
- $\mathcal{C} = \{(i,j)\mid T_{i,j}<\infty\}$ είναι το σύνολο όλων των επιτρεπτών ζευγών \(i,j).

##### Περιορισμός 2: 
Η ζήτηση του κάθε κέντρου διανομής πρέπει να καλύπτεται πλήρως. Ή μάλλον, για κάθε κέντρο, το άθροισμα των λαμβανόμενων ποσοτήτων από κάθε αποθήκη πρέπει να ισούται με την ζήτησή του. Συγκεκριμένα:
$$
\sum_{\;i : (i,j)\in \mathcal{C}} f_{i,j} \;=\; D_j,
\quad \forall\,j = 1,\ldots,12
$$

Υπαγορεύοντας στο Gurobi να ελαχιστοποιήσει το παραπάνω πρβλημα, παίρνουμε την βέλτιστη λύση. Στα αποτελέσματα βλεπουμε τον βέλτιστο δυνατό συνδυασμό αποθηκών-διαδρομών. To Gurobi μας ενημερώνει επίσης πως βρήκε δύο λύσεις, από τις οποίες κρατάμε την μικρότερη. Το κόστος είναι 17,929.23145933 (σε χιλιάδες $), όπως φαίνεται πάλι στην κονσόλα.

In [50]:
#We create our model.
model_2 = gp.Model('WarehouseLocation')

#Introducing binary decision variables Y. They correspond to opening a warehouse(or not -> 0).
y = model_2.addVars(warehouses , vtype = GRB.BINARY, name = 'open')

#We will works in tonnes of cargo. Thus, we introduce non-negative variables f.
#f[i,j] denotes flow of f[i,j] tonnes from warehouse i to center j. 
f = model_2.addVars(c.keys() , lb = 0.0 , name = 'flow')

#Constraint No. 1.
#For an open warehouse (thus the presence of y[i]), the output flow cannot exceed the 
#capacity.
#For closed warehouses it must ,of course, be zero.
#We iterate over warehouse list and set the flow f of cargo to a max of K[i] (corresponding capacity) for every
#center j.
for i in warehouses:
    model_2.addConstr(gp.quicksum(f[i , j] for j in centers if (i , j) in c) <= K[i] * y[i] , name=f"cap_{i}")

#Constraint No. 2.
#The distribution center demand must be satisfied.
#For every distribution center, the total flow coming from every warehouse must be equal to the 
#corresponding demand (D[j]). Very similar code structure as before.
#Iteration and summation is the way to go.
for j in centers:
    model_2.addConstr(gp.quicksum(f[i , j] for i in warehouses if (i , j) in c) == D[j] , name=f"demand_{j}")

#Note that the if statements in both constraint loops ensure feasibility.

#Now time to set our Objective function.
#We instruct Gurobi to:
#Minimize the sum of fixed operating costs plus the sum of per-tonne costs of cargo transport.
#Thus determine which warehouses to open (y) and how much cargo to transfer from i to j (f).
model_2.setObjective(gp.quicksum(F[i] * y[i] for i in warehouses)+ gp.quicksum(c[i,j] * f[i,j] for i , j in c),GRB.MINIMIZE)

#Aaaaand go!
model_2.optimize()

#Here we extract and print the results.
open_wh = [i for i in warehouses if y[i].X > 0.5]
print("Open Warehouses:", open_wh)

print("\nShipments (i→j in tonnes):")
for i , j in c:
    if f[i , j].X > 1e-6: #Exclude the noise from printing.
        print(f"  Warehouse {i} → Center {j}: {f[i , j].X : .1f} t")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 8840U w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 24 rows, 135 columns and 258 nonzeros
Model fingerprint: 0xe1ac8751
Variable types: 123 continuous, 12 integer (12 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+02]
  Objective range  [3e-01, 1e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 2e+02]
Found heuristic solution: objective 70816.666667
Presolve time: 0.00s
Presolved: 24 rows, 135 columns, 258 nonzeros
Variable types: 123 continuous, 12 integer (12 binary)

Root relaxation: objective 1.574011e+04, 25 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 15740.1111    0 